# Tính năng 3 – Trang **Thị trường việc làm**

Trang lọc tin tuyển dụng có công bố lương theo **nguồn**, **vai trò** và **khu vực**, rồi cho thấy kỹ năng nào được yêu cầu nhiều nhất và kỹ năng nào đi kèm mức lương cao. Trang chỉ mô tả dữ liệu, không dùng mô hình.

Dữ liệu là bộ salary-v2 (1.735 tin thật) giống trang Tổng quan; bộ lọc biến bức tranh chung thành câu trả lời riêng cho từng nghề, từng nơi.

Mã nguồn: `pages/market.py`, `src/career_market/ui/cache.py` (`filter_postings`), `src/career_market/ui/charts.py`.

In [ ]:
# Thiết lập: dùng đúng thư viện của ứng dụng (job_market/src), dữ liệu và mô hình đã lưu.
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "career_market").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from career_market.ui import charts  # thư viện biểu đồ chọn backend "Agg" (vẽ cho web)…
%matplotlib inline
# …nên bật lại chế độ inline để biểu đồ hiện trong notebook.
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [ ]:
from career_market.data.dataset import load_dataset

frame = load_dataset()


def filter_postings(sources=(), roles=(), regions=()):
    """Giống trang web: danh sách rỗng nghĩa là không lọc."""
    mask = pd.Series(True, index=frame.index)
    if sources:
        mask &= frame["source"].isin(sources)
    if roles:
        mask &= frame["role"].isin(roles)
    if regions:
        mask &= frame["region"].isin(regions)
    return frame[mask]


print(f"{len(frame):,} tin · {frame['source'].nunique()} nguồn · {frame['role'].nunique()} vai trò")

## 1. Kỹ năng theo bộ lọc
Quy ước: bộ lọc rỗng = không lọc; một tin chỉ tính một lần.

In [ ]:
cases = {
    "Toàn bộ": dict(),
    "Backend": dict(roles=["Backend Developer"]),
    "Data Engineer": dict(roles=["Data Engineer"]),
    "TP. Hồ Chí Minh": dict(regions=["TP. Hồ Chí Minh"]),
    "Chỉ ITviec": dict(sources=["IT Jobs 2026 (itviec)"]),
}
rows = []
for name, kwargs in cases.items():
    view = filter_postings(**kwargs)
    table = charts.skill_table(view).sort_values("postings", ascending=False)
    rows.append({
        "Bộ lọc": name,
        "Số tin": len(view),
        "Tin có kỹ năng": int((view["skill_count"] > 0).sum()),
        "Trung vị lương": view["salary"].median(),
        "Top 3 kỹ năng": ", ".join(table["skill"].head(3)) if len(table) else "—",
    })
pd.DataFrame(rows)

In [ ]:
view = filter_postings(roles=["Backend Developer"])
charts.top_skill_demand(view, limit=20)

## 2. Kỹ năng đi kèm lương cao
Thanh xám là kỹ năng có dưới 10 tin trong phần đã lọc: ít dữ liệu, không nên kết luận.

In [ ]:
charts.top_skill_salary(view, limit=20, min_postings=10)

⚠️ Nguồn TopCV 2026 **chỉ có chức danh**, không có mô tả công việc, nên kỹ năng được đoán từ tiêu đề: "tin có kỹ năng" thấp không có nghĩa là tin không đòi kỹ năng.

## Kết luận
- Bộ lọc biến bức tranh chung thành câu trả lời riêng cho từng nghề, từng nơi.
- Kỹ năng phổ biến chưa chắc là kỹ năng trả cao.
- Mức lương gắn với kỹ năng chỉ là tương quan, không chứng minh kỹ năng gây tăng lương.